# Explicit sklearn Logistic Plugin

This notebook keeps estimator code outside core Aegis and registers it explicitly before config validation. Train-mode YAML references a model ref with `source: plugin` and a stable model `id`.

In [ ]:
# ruff: noqa: E402, I001, UP035
from __future__ import annotations

import shutil
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Any, Mapping

def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'research').exists():
            return path
    raise RuntimeError('Run this notebook from inside the aegis-rd repository')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from research.aegis_research.component_registry import discover_component_registry
from research.aegis_research.config import resolve_lane_config
from research.aegis_research.experiments import run_experiment
from research.aegis_research.indicators import build_component_indicator_result
from research.aegis_research.labels import (
    LabelConfig,
    LabelGeneratorConfig,
    LabelTargetConfig,
    LabelTargetSelectionConfig,
    LabelTargetTransformConfig,
    build_label_result,
)
from research.aegis_research.model_contracts import (
    POSITIVE_CLASS_PROBABILITY,
    ModelDataset,
    ModelExecutionContext,
    ModelFitResult,
    ModelPluginDeclaration,
    ModelPluginDefinition,
    ModelPredictionResult,
)
from research.aegis_research.model_export import export_model_bundle
from research.aegis_research.model_registry import ModelRegistry

def write_component_registry(root: Path) -> Path:
    component_root = root / 'research' / 'components'
    indicator_root = component_root / 'indicators'
    label_root = component_root / 'labels'
    indicator_root.mkdir(parents=True, exist_ok=True)
    label_root.mkdir(parents=True, exist_ok=True)
    (indicator_root / 'returns.py').write_text('''
# %% define component metadata
COMPONENT_MANIFEST = {'family': 'indicators', 'id': 'example.returns', 'version': '1.0.0', 'input_names': ['Close'], 'param_names': [], 'output_names': ['returns'], 'default_outputs': ['returns'], 'default_model_features': [{'output': 'returns', 'transform': 'identity'}], 'supported_transforms': ['identity']}
COMPONENT_CALLABLE = 'run'

# %% main compute
def run(data):
    """Compute simple returns over the run-provided Close feature."""

    return data.feature('Close').pct_change().fillna(0.0)
'''.lstrip())
    (label_root / 'fixlb.py').write_text('''
# %% define component metadata
COMPONENT_MANIFEST = {'family': 'labels', 'id': 'example.fixlb', 'version': '1.0.0', 'input_names': ['Close'], 'target_role': 'supervised_target', 'target_kind': 'binary_classification', 'output_names': ['labels'], 'split_safety': {'purging_required': True}}
COMPONENT_CALLABLE = 'run'

# %% main compute
def run(data):
    """Placeholder component; the notebook supplies labels explicitly."""

    raise RuntimeError('this notebook passes an explicit label_result_builder')
'''.lstrip())
    return component_root


In [ ]:
class SklearnLogisticPlugin:
    def fit(self, dataset: ModelDataset, *, params: Mapping[str, Any], context: ModelExecutionContext) -> ModelFitResult:
        del context
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                max_iter=int(params.get('max_iter', 1000)),
                random_state=int(params.get('random_state', 42)),
            )),
        ])
        if dataset.target is None:
            raise ValueError('fit dataset target is required')
        model.fit(dataset.features, dataset.target.astype(int))
        classes = tuple(model.named_steps['classifier'].classes_)
        return ModelFitResult(
            state={'model': model},
            observed_classes=classes,
            class_probability_columns={class_label: f'class_{class_label}_probability' for class_label in classes},
            diagnostics={'estimator': 'sklearn.linear_model.LogisticRegression'},
            state_metadata={'state_format': 'pickle'},
        )

    def predict(self, state: Any, dataset: ModelDataset, *, params: Mapping[str, Any], context: ModelExecutionContext) -> ModelPredictionResult:
        del params, context
        model = state['model']
        classes = tuple(model.named_steps['classifier'].classes_)
        columns = {class_label: f'class_{class_label}_probability' for class_label in classes}
        return ModelPredictionResult(
            probabilities=pd.DataFrame(
                model.predict_proba(dataset.features),
                index=dataset.row_index,
                columns=[columns[class_label] for class_label in classes],
            ),
            observed_classes=classes,
            class_probability_columns=columns,
        )


In [ ]:
def validate_params(params: Mapping[str, Any]) -> Mapping[str, str]:
    issues = {}
    if 'max_iter' in params and (not isinstance(params['max_iter'], int) or params['max_iter'] <= 0):
        issues['max_iter'] = 'must be a positive integer'
    if 'random_state' in params and not isinstance(params['random_state'], int):
        issues['random_state'] = 'must be an integer'
    return issues

def build_registry():
    registry = ModelRegistry()
    registry.register(ModelPluginDefinition(
        declaration=ModelPluginDeclaration(
            id='examples.sklearn_logistic',
            version='1.0.0',
            prediction_outputs=(POSITIVE_CLASS_PROBABILITY,),
            state_schema_version='example_sklearn_logistic_state.v1',
            package_versions={'sklearn': sklearn.__version__},
        ),
        plugin=SklearnLogisticPlugin(),
        validate_params=validate_params,
    ))
    return registry.freeze()


In [ ]:
scratch = TemporaryDirectory(prefix='aegis-model-plugin-')
scratch_root = Path(scratch.name)
output_dir = Path('runs') / scratch_root.name
component_registry = discover_component_registry(
    root=write_component_registry(scratch_root),
    repo_root=scratch_root,
)
registry = build_registry()
experiment = {
    'schema_version': 4,
    'lane': 'train',
    'name': 'sklearn_logistic_plugin_example',
    'output_dir': output_dir.as_posix(),
    'data': {
        'source': 'synthetic',
        'symbols': ['SYN'],
        'start': '2020-01-01',
        'timeframe': '1D',
        'rows': 240,
        'seed': 42,
        'arrays': ['OHLCV'],
    },
    'indicators': [{'source': 'component', 'ids': ['example.returns']}],
    'labeler': {'id': 'example.fixlb'},
    'train': {
        'model': {
            'source': 'plugin',
            'id': 'examples.sklearn_logistic',
            'min_train_samples': 50,
            'params': {'max_iter': 1000, 'random_state': 42},
        },
        'split': {'kind': 'purged_kfold', 'n_folds': 3, 'n_test_folds': 1, 'max_splits': 3},
        'signals': {'policy': 'long_only_hysteresis', 'long_entry_threshold': 0.55, 'long_exit_threshold': 0.50, 'execution_timing': 'next_open'},
    },
    'portfolio': {'entry_budget': 1.0, 'direction': 'longonly'},
    'report': {'freq': '1D', 'year_freq': '252D', 'min_oos_sharpe': 0.5, 'max_oos_drawdown': 0.35, 'min_oos_trades': 1},
}
resolved = resolve_lane_config(
    experiment,
    component_registry=component_registry,
    model_registry=registry,
    expected_lane='train',
)
label_config = LabelConfig(
    generator=LabelGeneratorConfig(params={'n': 5}),
    target=LabelTargetConfig(
        select=LabelTargetSelectionConfig(params={'n': 5}),
        transform=LabelTargetTransformConfig(params={'threshold': 0.0}),
    ),
)

def label_result_builder(data):
    return build_label_result(data.feature('Close'), label_config, high=data.feature('High'), low=data.feature('Low'))

def indicator_result_builder(data):
    return build_component_indicator_result(
        data,
        resolved.config.indicators,
        component_registry=component_registry,
    )

result = run_experiment(
    resolved,
    label_result_builder=label_result_builder,
    indicator_result_builder=indicator_result_builder,
)
result['status']


In [ ]:
# Optional producer-side export for another prediction-only runtime project.
# The consuming project must register reviewed plugin code and validate this metadata before loading native state.
export_model_bundle(
    result['run_dir'],
    model_artifact_id='validation.split_0.model',
    output_dir=Path(result['run_dir']) / 'exports' / 'split_0_model',
)


In [ ]:
scratch.cleanup()
shutil.rmtree(output_dir, ignore_errors=True)
